# 04 - Tool Use com APIs Externas
> Conectando Claude a APIs reais

**Modulo:** `03_tool_use` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import httpx, json

TOOLS=[
    {'name':'buscar_cep','description':'Busca endereco por CEP.',
     'input_schema':{'type':'object','properties':{'cep':{'type':'string'}},'required':['cep']}},
    {'name':'cotacao','description':'Cotacao de moeda em BRL.',
     'input_schema':{'type':'object','properties':{'moeda':{'type':'string'}},'required':['moeda']}}
]

def buscar_cep(cep):
    r=httpx.get(f'https://viacep.com.br/ws/{cep.replace("-","")}/json/',timeout=5)
    d=r.json()
    return 'CEP invalido' if 'erro' in d else f'{d["logradouro"]}, {d["localidade"]}-{d["uf"]}'

def cotacao(moeda):
    r=httpx.get(f'https://economia.awesomeapi.com.br/json/last/{moeda}-BRL',timeout=5)
    d=r.json(); k=f'{moeda}BRL'
    return f'{moeda}/BRL: R${float(d[k]["bid"]):.2f}' if k in d else 'Nao encontrado'

FUNS={'buscar_cep':buscar_cep,'cotacao':cotacao}

def agente(q):
    msgs=[{'role':'user','content':q}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=TOOLS,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':FUNS[b.name](**b.input)})
        msgs.append({'role':'user','content':res})

print(agente('Qual o endereco do CEP 01310-100 e a cotacao do dolar?'))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
